# Práctica — Clase 7. Deep Q-Networks (DQN)

**DIH DATalife — Curso de Reinforcement Learning (RL) desde cero**

En esta práctica trabajas sobre el mismo DQN (Deep Q-Network) y el mismo entorno CartPole de la clase. Dos ejercicios:

1. Completas tú misma o tú mismo el cálculo de la pérdida de DQN, la pieza que viste en el bloque tres de la clase
2. Programas un decaimiento de epsilon y comparas su curva de recompensa con la del epsilon fijo que usaste en clase

**Antes de empezar, ejecuta las celdas siguientes para tener listo el andamiaje.** Contienen la instalación de `gymnasium`, las importaciones, los hiperparámetros, la red Q, el buffer de repetición y la política épsilon-greedy: son idénticos a los de la clase, así que no hace falta que los edites.


In [ ]:
# Instalamos gymnasium: es la UNICA dependencia externa de esta practica.
# PyTorch, numpy y matplotlib ya vienen preinstalados en Google Colab.
!pip install gymnasium


In [ ]:
import random
from collections import deque, namedtuple

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import gymnasium as gym

SEMILLA = 42
random.seed(SEMILLA)
np.random.seed(SEMILLA)
torch.manual_seed(SEMILLA)

# --- Hiperparametros (identicos a los de la clase) ---
NOMBRE_ENTORNO = "CartPole-v1"
NUM_EPISODIOS = 200                    # episodios de entrenamiento
GAMMA = 0.99                           # factor de descuento
TASA_APRENDIZAJE = 1e-3
TAMANO_BUFFER = 10000                  # capacidad del buffer de repeticion
TAMANO_LOTE = 64                       # tamano del mini-lote muestreado en cada paso
PASOS_ACTUALIZAR_RED_OBJETIVO = 500    # cada cuantos pasos se sincroniza la red objetivo
PASOS_MINIMOS_ANTES_ENTRENAR = 1000    # esperamos a tener algo de experiencia antes de entrenar


In [ ]:
class RedQ(nn.Module):
    """Aproximador de funciones: recibe un estado y devuelve un valor
    estimado para cada accion posible (identica a la de la clase)."""

    def __init__(self, dim_estado, num_acciones):
        super().__init__()
        self.capas = nn.Sequential(
            nn.Linear(dim_estado, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, num_acciones),
        )

    def forward(self, estado):
        return self.capas(estado)


Transicion = namedtuple(
    "Transicion", ["estado", "accion", "recompensa", "siguiente_estado", "terminado"]
)


class BufferRepeticion:
    """Buffer de repeticion de experiencias, identico al de la clase."""

    def __init__(self, capacidad):
        self.memoria = deque(maxlen=capacidad)

    def guardar(self, *args):
        self.memoria.append(Transicion(*args))

    def muestrear(self, tamano_lote):
        return random.sample(self.memoria, tamano_lote)

    def __len__(self):
        return len(self.memoria)


def elegir_accion(red_online, estado, epsilon, num_acciones):
    """Politica epsilon-greedy, identica a la de la clase."""
    if random.random() < epsilon:
        return random.randrange(num_acciones)
    with torch.no_grad():
        estado_tensor = torch.tensor(estado, dtype=torch.float32).unsqueeze(0)
        valores_q = red_online(estado_tensor)
        return int(torch.argmax(valores_q, dim=1).item())


entorno = gym.make(NOMBRE_ENTORNO)
DIM_ESTADO = entorno.observation_space.shape[0]
NUM_ACCIONES = entorno.action_space.n
entorno.close()

print("Dimension del estado:", DIM_ESTADO, "| Numero de acciones:", NUM_ACCIONES)


## Ejercicio 1 — Completa la pérdida de DQN desde cero

En la clase viste la fórmula de la pérdida de DQN:

`L(θ) = E[ (r + γ · max_a' Q(s',a';θ⁻) − Q(s,a;θ))² ]`

Abajo tienes `entrenar_paso` con el andamiaje ya construido: la preparación del mini-lote en tensores y el cálculo de `valores_q` (la predicción, `Q(s,a;θ)`) ya están completos. **Tu tarea es rellenar los dos `# TODO`:**

1. Calcula `objetivo`, el valor r + γ · max_a' Q(s',a';θ⁻), usando `red_objetivo` dentro de un bloque `with torch.no_grad():` (el "stop-gradient": el error nunca debe fluir hacia la red objetivo)
2. Calcula `perdida`, el error cuadrático medio (MSE) entre `valores_q` (la predicción) y `objetivo`

**Pistas**

- `red_objetivo(siguientes_estados)` te da, para cada transición del lote, el valor Q de las dos acciones en el estado siguiente; con `.max(dim=1)[0]` te quedas con el mayor de los dos, es decir, `max_a' Q(s',a';θ⁻)`
- Cuando `terminado` vale uno (el episodio acabó), no hay estado siguiente que valga la pena mirar: multiplica el término `GAMMA * max_q_siguiente` por `(1 - terminados)` para anularlo en ese caso
- `nn.functional.mse_loss(prediccion, objetivo)` calcula el error cuadrático medio entre dos tensores

Si dejas los `# TODO` sin completar, la celda de prueba de más abajo va a lanzar un error: es la señal de que te falta terminar el cálculo.


In [ ]:
def entrenar_paso(red_online, red_objetivo, buffer, optimizador):
    """Un paso de entrenamiento de DQN: muestrea un mini-lote del buffer y
    ajusta la red online para acercar su prediccion al objetivo calculado
    con la red objetivo (target network)."""
    lote = buffer.muestrear(TAMANO_LOTE)
    estados = torch.tensor(np.array([t.estado for t in lote]), dtype=torch.float32)
    acciones = torch.tensor([t.accion for t in lote], dtype=torch.int64).unsqueeze(1)
    recompensas = torch.tensor([t.recompensa for t in lote], dtype=torch.float32)
    siguientes_estados = torch.tensor(
        np.array([t.siguiente_estado for t in lote]), dtype=torch.float32
    )
    terminados = torch.tensor([t.terminado for t in lote], dtype=torch.float32)

    # Q(s,a;theta): valor que la red online asigna a la accion que de verdad se tomo
    # (ya completo: no hace falta tocarlo)
    valores_q = red_online(estados).gather(1, acciones).squeeze(1)

    # TODO 1: calcula "objetivo" = r + gamma * max_a' Q(s',a';theta-), usando
    # red_objetivo dentro de un bloque `with torch.no_grad():`. Recuerda anular
    # el segundo termino cuando el episodio termino, multiplicando por (1 - terminados).
    # objetivo = ...

    # TODO 2: calcula "perdida" como el error cuadratico medio (MSE) entre
    # valores_q (la prediccion) y objetivo.
    # perdida = ...

    optimizador.zero_grad()
    perdida.backward()
    optimizador.step()


**Ejecuta la celda siguiente para entrenar con tu `entrenar_paso` y comprobar si funciona.** Usa el mismo epsilon fijo de la clase (0,1). Si tu implementación es correcta, la recompensa media de los últimos 20 episodios debería subir con claridad hacia el final del entrenamiento, igual que viste en clase.


In [ ]:
EPSILON_EJ1 = 0.1  # mismo epsilon fijo que usaste en clase

random.seed(SEMILLA)
np.random.seed(SEMILLA)
torch.manual_seed(SEMILLA)

entorno = gym.make(NOMBRE_ENTORNO)
red_online = RedQ(DIM_ESTADO, NUM_ACCIONES)
red_objetivo = RedQ(DIM_ESTADO, NUM_ACCIONES)
red_objetivo.load_state_dict(red_online.state_dict())
red_objetivo.eval()

optimizador = optim.Adam(red_online.parameters(), lr=TASA_APRENDIZAJE)
buffer = BufferRepeticion(TAMANO_BUFFER)

recompensas_ej1 = []
pasos_totales = 0

for episodio in range(NUM_EPISODIOS):
    estado, _ = entorno.reset(seed=SEMILLA + episodio)
    recompensa_episodio = 0.0
    terminado = False
    truncado = False

    while not (terminado or truncado):
        accion = elegir_accion(red_online, estado, EPSILON_EJ1, NUM_ACCIONES)
        siguiente_estado, recompensa, terminado, truncado, _ = entorno.step(accion)

        buffer.guardar(estado, accion, recompensa, siguiente_estado, terminado)
        estado = siguiente_estado
        recompensa_episodio += recompensa
        pasos_totales += 1

        if len(buffer) >= PASOS_MINIMOS_ANTES_ENTRENAR:
            entrenar_paso(red_online, red_objetivo, buffer, optimizador)

        if pasos_totales % PASOS_ACTUALIZAR_RED_OBJETIVO == 0:
            red_objetivo.load_state_dict(red_online.state_dict())

    recompensas_ej1.append(recompensa_episodio)

    if (episodio + 1) % 20 == 0:
        print(
            f"Episodio {episodio + 1}/{NUM_EPISODIOS} "
            f"| recompensa media (ultimos 20)={np.mean(recompensas_ej1[-20:]):.1f}"
        )

entorno.close()

plt.figure(figsize=(10, 5))
plt.plot(recompensas_ej1, alpha=0.4, label="Recompensa por episodio")
media_movil = np.convolve(recompensas_ej1, np.ones(20) / 20, mode="valid")
plt.plot(range(19, len(recompensas_ej1)), media_movil, label="Media movil (20 episodios)")
plt.xlabel("Episodio")
plt.ylabel("Recompensa total")
plt.title("Ejercicio 1: tu perdida de DQN, con epsilon fijo")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


<details>
<summary>💡 Ver solución</summary>

```python
def entrenar_paso(red_online, red_objetivo, buffer, optimizador):
    """Un paso de entrenamiento de DQN: muestrea un mini-lote del buffer y
    ajusta la red online para acercar su prediccion al objetivo calculado
    con la red objetivo (target network)."""
    lote = buffer.muestrear(TAMANO_LOTE)
    estados = torch.tensor(np.array([t.estado for t in lote]), dtype=torch.float32)
    acciones = torch.tensor([t.accion for t in lote], dtype=torch.int64).unsqueeze(1)
    recompensas = torch.tensor([t.recompensa for t in lote], dtype=torch.float32)
    siguientes_estados = torch.tensor(
        np.array([t.siguiente_estado for t in lote]), dtype=torch.float32
    )
    terminados = torch.tensor([t.terminado for t in lote], dtype=torch.float32)

    # Q(s,a;theta): valor que la red online asigna a la accion que de verdad se tomo
    valores_q = red_online(estados).gather(1, acciones).squeeze(1)

    # Objetivo: r + gamma * max_a' Q(s',a';theta-), calculado con la red objetivo.
    # torch.no_grad() implementa el "stop-gradient": el error se propaga
    # solo hacia la red online, nunca hacia la red objetivo.
    with torch.no_grad():
        max_q_siguiente = red_objetivo(siguientes_estados).max(dim=1)[0]
        objetivo = recompensas + GAMMA * max_q_siguiente * (1 - terminados)

    # Perdida: error cuadratico medio entre la prediccion y el objetivo
    perdida = nn.functional.mse_loss(valores_q, objetivo)

    optimizador.zero_grad()
    perdida.backward()
    optimizador.step()
```

El punto clave es el `with torch.no_grad():` alrededor del cálculo de `objetivo`: sin él, PyTorch propagaría gradiente también hacia `red_objetivo`, y estarías moviendo el propio objetivo para que se acercase a tu predicción, en vez de al revés.
</details>


## Ejercicio 2 — Programa el decaimiento de epsilon y compáralo con el epsilon fijo

En clase usaste un epsilon **fijo** (0,1) durante todo el entrenamiento, para que el código en vivo fuera más sencillo de seguir. El manual recomienda, en cambio, un epsilon que **decae**: empieza explorando mucho (epsilon cercano a uno) y termina explorando poco (epsilon cercano a 0,05), a medida que la red aprende.

**Para que puedas avanzar aunque el ejercicio uno no te haya quedado terminado**, esta celda incluye una copia ya completa y funcionando de `entrenar_paso`. Tu tarea es otra: completar la función `calcular_epsilon`, donde pone `# TODO`, con la fórmula de decaimiento lineal.

**Tu tarea**

1. Calcula `progreso`, la fracción del decaimiento ya recorrida: el episodio actual dividido entre `EPISODIOS_DECAIMIENTO`, sin superar nunca uno
2. Calcula `epsilon` interpolando linealmente entre `EPSILON_INICIAL` y `EPSILON_FINAL`, según ese progreso

**Pistas**

- `progreso = min(episodio / EPISODIOS_DECAIMIENTO, 1.0)`
- La interpolación lineal entre dos valores `A` y `B` según una fracción `p` es `A + p * (B - A)`: cuando `p` vale cero devuelve `A`, y cuando `p` vale uno devuelve `B`
- Aquí `A` es `EPSILON_INICIAL` y `B` es `EPSILON_FINAL`


In [ ]:
def entrenar_paso(red_online, red_objetivo, buffer, optimizador):
    """Copia completa del paso de entrenamiento de DQN (identica a la solucion
    del ejercicio 1), para que puedas avanzar con este ejercicio de todos modos."""
    lote = buffer.muestrear(TAMANO_LOTE)
    estados = torch.tensor(np.array([t.estado for t in lote]), dtype=torch.float32)
    acciones = torch.tensor([t.accion for t in lote], dtype=torch.int64).unsqueeze(1)
    recompensas = torch.tensor([t.recompensa for t in lote], dtype=torch.float32)
    siguientes_estados = torch.tensor(
        np.array([t.siguiente_estado for t in lote]), dtype=torch.float32
    )
    terminados = torch.tensor([t.terminado for t in lote], dtype=torch.float32)

    valores_q = red_online(estados).gather(1, acciones).squeeze(1)

    with torch.no_grad():
        max_q_siguiente = red_objetivo(siguientes_estados).max(dim=1)[0]
        objetivo = recompensas + GAMMA * max_q_siguiente * (1 - terminados)

    perdida = nn.functional.mse_loss(valores_q, objetivo)

    optimizador.zero_grad()
    perdida.backward()
    optimizador.step()


EPSILON_INICIAL = 1.0
EPSILON_FINAL = 0.05
EPISODIOS_DECAIMIENTO = 120    # episodios en los que epsilon baja de inicial a final


def calcular_epsilon(episodio):
    """Decaimiento lineal de epsilon: empieza en EPSILON_INICIAL y llega a
    EPSILON_FINAL en el episodio EPISODIOS_DECAIMIENTO, sin bajar mas alla."""
    # TODO 1: calcula "progreso", la fraccion del decaimiento ya recorrida
    # progreso = ...

    # TODO 2: interpola linealmente entre EPSILON_INICIAL y EPSILON_FINAL
    # epsilon = ...

    return epsilon


**Ejecuta la celda siguiente.** Entrena dos agentes con el mismo `entrenar_paso` y las mismas semillas: uno con tu `calcular_epsilon` (decaimiento) y otro con el epsilon fijo de la clase (0,1), para comparar ambas curvas de recompensa en la misma gráfica.


In [ ]:
def entrenar_dqn(funcion_epsilon):
    """Entrena un DQN completo y devuelve la recompensa por episodio.
    `funcion_epsilon(episodio)` decide el epsilon de cada episodio."""
    random.seed(SEMILLA)
    np.random.seed(SEMILLA)
    torch.manual_seed(SEMILLA)

    entorno = gym.make(NOMBRE_ENTORNO)
    red_online = RedQ(DIM_ESTADO, NUM_ACCIONES)
    red_objetivo = RedQ(DIM_ESTADO, NUM_ACCIONES)
    red_objetivo.load_state_dict(red_online.state_dict())
    red_objetivo.eval()

    optimizador = optim.Adam(red_online.parameters(), lr=TASA_APRENDIZAJE)
    buffer = BufferRepeticion(TAMANO_BUFFER)

    recompensas = []
    pasos_totales = 0

    for episodio in range(NUM_EPISODIOS):
        estado, _ = entorno.reset(seed=SEMILLA + episodio)
        recompensa_episodio = 0.0
        terminado = False
        truncado = False
        epsilon = funcion_epsilon(episodio)

        while not (terminado or truncado):
            accion = elegir_accion(red_online, estado, epsilon, NUM_ACCIONES)
            siguiente_estado, recompensa, terminado, truncado, _ = entorno.step(accion)

            buffer.guardar(estado, accion, recompensa, siguiente_estado, terminado)
            estado = siguiente_estado
            recompensa_episodio += recompensa
            pasos_totales += 1

            if len(buffer) >= PASOS_MINIMOS_ANTES_ENTRENAR:
                entrenar_paso(red_online, red_objetivo, buffer, optimizador)

            if pasos_totales % PASOS_ACTUALIZAR_RED_OBJETIVO == 0:
                red_objetivo.load_state_dict(red_online.state_dict())

        recompensas.append(recompensa_episodio)

    entorno.close()
    return recompensas


print("Entrenando con epsilon fijo (el de la clase)...")
recompensas_fijo = entrenar_dqn(lambda episodio: 0.1)

print("Entrenando con tu decaimiento de epsilon...")
recompensas_decay = entrenar_dqn(calcular_epsilon)

print("Recompensa media, ultimos 20 episodios, epsilon fijo:", np.mean(recompensas_fijo[-20:]))
print("Recompensa media, ultimos 20 episodios, epsilon con decaimiento:", np.mean(recompensas_decay[-20:]))


In [ ]:
def media_movil(valores, ventana=20):
    return np.convolve(valores, np.ones(ventana) / ventana, mode="valid")

plt.figure(figsize=(10, 5))
plt.plot(range(19, len(recompensas_fijo)), media_movil(recompensas_fijo), label="Epsilon fijo (0.1, el de la clase)")
plt.plot(range(19, len(recompensas_decay)), media_movil(recompensas_decay), label="Epsilon con decaimiento (el tuyo)")
plt.xlabel("Episodio")
plt.ylabel("Recompensa total (media movil de 20 episodios)")
plt.title("Ejercicio 2: epsilon fijo vs. epsilon con decaimiento")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


<details>
<summary>💡 Ver solución</summary>

```python
def calcular_epsilon(episodio):
    """Decaimiento lineal de epsilon: empieza en EPSILON_INICIAL y llega a
    EPSILON_FINAL en el episodio EPISODIOS_DECAIMIENTO, sin bajar mas alla."""
    # 1. Fraccion del decaimiento ya recorrida, sin superar nunca uno
    progreso = min(episodio / EPISODIOS_DECAIMIENTO, 1.0)

    # 2. Interpolacion lineal entre EPSILON_INICIAL y EPSILON_FINAL
    epsilon = EPSILON_INICIAL + progreso * (EPSILON_FINAL - EPSILON_INICIAL)

    return epsilon
```

Con esta fórmula, `epsilon` vale `EPSILON_INICIAL` en el episodio cero y desciende de forma lineal hasta `EPSILON_FINAL` en el episodio `EPISODIOS_DECAIMIENTO`; a partir de ahí, `progreso` se queda fijo en uno y `epsilon` ya no baja más.

**Sobre la comparación**: no des por hecho que una curva "gana" a la otra de forma definitiva con solo una repetición del experimento. CartPole es un entorno pequeño y el resultado depende bastante de la semilla. Lo importante es que entiendas el mecanismo: un epsilon que decae explora mucho al principio, cuando la red todavía no sabe nada, y explota más adelante, cuando ya aprendió algo útil. Un epsilon fijo y bajo, en cambio, explora poco desde el minuto uno, lo que puede retrasar el momento en que el agente descubre las trayectorias buenas.
</details>


## Para seguir practicando

Vuelve al notebook de clase y aplica las ideas de "Ponlo en práctica" sobre tu propia versión: **cambia `PASOS_ACTUALIZAR_RED_OBJETIVO`** entre 50, 500 y 5.000, o **cambia `TAMANO_BUFFER`** entre 500 y 50.000, y observa cómo se transforma la curva de recompensa en cada caso. Coméntalo con tu equipo: ¿qué combinación de hiperparámetros entrena de forma más estable?
